# 2 · OIE extraction with Gemini  (per article)

Free-form open information extraction on one article at a time — **entities + types**: the model
invents the type label that best fits each entity, then defines each distinct type. Canonicalizing
those labels to schema.org is nb4's job; extracting the relations between them is nb6's.

- **Config:** one line — `EXPERIMENT`; everything else comes from `configs/<EXPERIMENT>.json`.
- **Input:** `dataset/articles/<id>.txt`.
- **Output:** `<id>/two_staged/1_entities/oie_entities<_osfx>.json`.

Articles are independent here, so the batch cell runs `WORKERS` of them at once.

In [1]:
# --- resolve Phase2_LLM_Extraction root (works whether this notebook lives at the
# project root or under notebooks/<track>/) so results/ & dataset/ paths are stable.
from pathlib import Path
def _find_root(start: Path = None):
    start = (start or Path.cwd()).resolve()
    for p in (start, *start.parents):
        if (p / 'results').is_dir() and (p / 'dataset').is_dir():
            return p
    return start
ROOT = _find_root()
# make Path writes create missing parent dirs (results/ now has sub-folders like
# two_staged/1_entities/); guarded so re-running the cell never double-wraps.
import pathlib as _pl
if not getattr(_pl.Path.write_text, '_mkparents', False):
    _wt, _wb = _pl.Path.write_text, _pl.Path.write_bytes
    def write_text(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wt(self, *a, **k)
    def write_bytes(self, *a, **k):
        self.parent.mkdir(parents=True, exist_ok=True); return _wb(self, *a, **k)
    write_text._mkparents = write_bytes._mkparents = True
    _pl.Path.write_text, _pl.Path.write_bytes = write_text, write_bytes

In [2]:
# ---- WHICH EXPERIMENT ---------------------------------------------------------------------
# ONE name decides the whole run. Every knob below comes from configs/<EXPERIMENT>.json — the SAME
# file every other notebook in both tracks reads, so a chain cannot drift apart stage to stage.
#   list what exists :  python -m experiment_config list          (run from notebooks/)
#   see one resolved :  python -m experiment_config show main
# $PHASE2_EXPERIMENT wins when it is set, so a headless batch of ablations (run_e2e.sh) can sweep
# experiments without editing a single notebook.
import os
EXPERIMENT = os.environ.get("PHASE2_EXPERIMENT") or "main"

import sys, pathlib, importlib
for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_p / "experiment_config.py").exists():
        sys.path.insert(0, str(_p)); break
import experiment_config as xcfg; importlib.reload(xcfg)

# Sets RESULTS_MODEL, BACKEND, GEMINI_MODEL, RUN_TAG, SELECTED_ARTICLES and this section's own
# knobs: WORKERS, THINKING_BUDGET, MAX_OUTPUT_TOKENS.
CFG = xcfg.load(EXPERIMENT, globals(), "2")
# A one-off try goes RIGHT HERE, below the load, where it wins over the file.

# --- derived paths -------------------------------------------------------------------------
from pathlib import Path

DATASET_DIR = ROOT / "dataset/articles"          # inputs:  <id>.txt
RESULTS_DIR = ROOT / "results" / RESULTS_MODEL

# _osfx tags everything this notebook writes, so a new configuration never overwrites an older run;
# nb4 reads the tag back as its own INPUT_TAG. There is no VARIANT on this side of the staged track:
# VARIANT names a reification prompt arm, and entity extraction has only one.
_osfx = f"_{RUN_TAG}" if RUN_TAG else ""
OUTPUT_NAME = f"two_staged/1_entities/oie_entities{_osfx}.json"

_available = sorted((p.stem for p in DATASET_DIR.glob("*.txt")), key=xcfg.article_key)
if SELECTED_ARTICLES is None:
    ARTICLE_IDS = _available
else:
    missing = [a for a in SELECTED_ARTICLES if a not in _available]
    if missing:
        print("WARNING: skipping (no .txt in dataset):", missing)
    ARTICLE_IDS = [a for a in SELECTED_ARTICLES if a in _available]
assert ARTICLE_IDS, "No runnable articles (need dataset/articles/<id>.txt). Check the config."
ARTICLE_ID = ARTICLE_IDS[0]                      # the single article the demo cells below use

print(f"\n{len(_available)} article texts available; this run processes {len(ARTICLE_IDS)}"
      f" -> results/{RESULTS_MODEL}/<id>/{OUTPUT_NAME}")

experiment 'main'  ·  section 2 (two_staged · entity extraction)
----------------------------------------------------------------
  The reported configuration: schema-only inline retrieval + global re-consolidation, cross-encoder rerank on the relation side, two-call consolidation with generalize. Every ablation is measured against it.
----------------------------------------------------------------
  RESULTS_MODEL                'llama33-70b'
  BACKEND                      'llama33-70b'
  VARIANT                      'dedup'
  RUN_TAG                      'main_val_r1'
  INPUT_TAG                    'main_val_r1'
  REPEAT                       1
  SPLIT                        'validation'
  SELECTED_ARTICLES            '[20 articles: art1 … art74]'
  MATCH_MODEL                  'intfloat/e5-base-v2'
  GEMINI_MODEL                 'gemini-2.5-flash'
  ····························
  WORKERS                      4
  THINKING_BUDGET              -1
  MAX_OUTPUT_TOKENS            None
  r

In [3]:
# --- Gemini client -----------------------------------------------------------------------
%pip install -q google-genai

import os, json, time
from google import genai
from google.genai import types as genai_types
from IPython.display import Markdown, display

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Set GEMINI_API_KEY (or GOOGLE_API_KEY) in the env and restart the kernel."

client = genai.Client(api_key=GEMINI_API_KEY)
last_latency_s = None
LAST_USAGE = None

_RETRY_STATUS = (429, 500, 502, 503, 504)
_RETRY_WORDS = ("UNAVAILABLE", "RESOURCE_EXHAUSTED", "INTERNAL", "high demand", "overloaded")


def _gemini_retryable(exc) -> bool:
    """A transient server-side failure, as opposed to a bad request worth not repeating."""
    code = getattr(exc, "code", None) or getattr(exc, "status_code", None)
    if code in _RETRY_STATUS:
        return True
    return any(w.lower() in str(exc).lower() for w in _RETRY_WORDS)


def _gemini_call_with_retry(fn, attempts=5, base=4):
    """Call `fn()`, retrying TRANSIENT API errors with backoff (4s, 8s, 16s, 32s).

    Without this one `503 UNAVAILABLE - This model is currently experiencing high demand` ends the
    whole stage: it killed nb4 two articles into twenty and took nb6b and nb7b down with it. Google's
    own message says such spikes are temporary, which is what a backoff is for. A 400 or 403 still
    fails at once — repeating a malformed or unauthorised request achieves nothing.
    """
    import time
    for attempt in range(attempts):
        try:
            return fn()
        except Exception as exc:                       # noqa: BLE001 - re-raised below
            if attempt == attempts - 1 or not _gemini_retryable(exc):
                raise
            wait = min(60, base * 2 ** attempt)
            code = getattr(exc, "code", None) or getattr(exc, "status_code", "?")
            print(f"    [retry {attempt + 1}/{attempts - 1}] {type(exc).__name__} {code} — "
                  f"waiting {wait}s", flush=True)
            time.sleep(wait)


def gemini_json(system_prompt, user_message, model=None, temperature=0):
    """Call Gemini with JSON-mode output; return the parsed object, recording tokens + latency."""
    global last_latency_s, LAST_USAGE
    t0 = time.perf_counter()
    resp = _gemini_call_with_retry(lambda: client.models.generate_content(
        model=model or GEMINI_MODEL,
        contents=user_message,
        config=genai_types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=temperature,
            response_mime_type="application/json",
        ),
    ))
    last_latency_s = time.perf_counter() - t0
    LAST_USAGE = nb_parallel.record_gemini_usage(resp, last_latency_s)   # prompt/output/THINKING tokens + latency
    return json.loads(resp.text)


# --- select the backend ---------------------------------------------------------------------
# Must stay LAST in this cell: the Gemini `gemini_json` above is redefined every time the cell
# runs, so whichever binding happens last wins. In a separate cell this would be silently undone.
# activate() sets gemini_json, GEMINI_MODEL, RESULTS_DIR, MAX_OUTPUT_TOKENS and THINKING_BUDGET.
import sys, pathlib, importlib
for _p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_p / "llm_backend.py").exists():
        sys.path.insert(0, str(_p)); break
import nb_parallel, llm_backend
importlib.reload(nb_parallel); importlib.reload(llm_backend)   # plain `import` is a no-op once cached
nb_parallel.install()           # thread-aware stdout + the token/latency ledger used by usage_report

llm_backend.activate(BACKEND, ROOT, globals(), results=RESULTS_MODEL)


Note: you may need to restart the kernel to use updated packages.
backend: vLLM http://localhost:8000/v1 model=llama33-70b thinking=False structured=True max_tokens=None temperature=caller top_p=None seed=None timeout=3600s  ->  results/llama33-70b/


{'backend': 'llama33-70b',
 'kind': 'vllm',
 'model': 'llama33-70b',
 'thinking': False,
 'structured': True,
 'results_dir': PosixPath('/Users/marwah/Documents/Thesis/TriplesExtractionFromWebLLMs/Phase2_LLM_Extraction/results/llama33-70b')}

## 1. Entity + type extraction

In [4]:
# Free-form entity + type extraction -> results/<id>/<OUTPUT_NAME>
OIE_ENTITY_SYSTEM_PROMPT = """You are an expert Entity Recognition and Typing system.
You are given a TEXT to analyze, and a REFERENCE_DATE (the article's publication date, ISO-8601) to anchor relative time expressions.
TASK: Extract every entity explicitly mentioned in the TEXT. For each entity, assign the single most appropriate type label, choosing the label freely based on what the entity is. You are not given a fixed set of types: invent whatever type label best describes each entity. The same surface text may be extracted more than once if, in different parts of the text, it clearly refers to different kinds of things (i.e. warrants different type labels). After extraction, produce a definition for every unique type label you used.
RULES:
1. Extract entities exactly as they appear; preserve original casing and punctuation in the "text" field. Do not invent, infer, or normalize the surface text of entities that are not explicitly mentioned. (Any resolved/normalized value goes in a separate field, never in "text" — see rule 7.)
2. Assign each entity a free-form type label of your own choosing. Pick the label that most precisely captures what kind of thing the entity is, given the evidence in the text. Keep the type a REUSABLE class: do not embed a specific named entity/instance in the label (use the general class, never a label containing a specific proper name). Do not restrict yourself to any predefined vocabulary.
3. Keep type labels consistent: if two entities are the same kind of thing, give them the exact same label (same spelling, same casing). Prefer a single clear noun or short noun phrase per label (e.g. "Person", "City", "Software Library", "Legal Case").
4. Choose the most specific label that still names a reusable category — one general enough to also apply to other entities of the same kind, not a label so narrow it fits essentially this entity alone. Favor a stable, commonly-recognized class over a hyper-specific or one-off label unlikely to recur across documents. Stay grounded in the text: do not over-generalize, and do not assert a specificity the evidence does not support.
5. Deduplicate on the (text, type) pair, not on text alone:
   - If the same surface text appears multiple times and refers to the same kind of thing each time, include it once.
   - If the same surface text refers to genuinely different kinds of things in different places (each supported by the text), include it once per distinct type, using the same surface text with the different type labels. Only do this when the evidence clearly supports more than one type for that exact mention text; do not split a single referent across types just because a label is uncertain.
6. "confidence" is your certainty in the assigned type for this mention, a value between 0 and 1. Set it per entity from how strongly the text supports the type — lower it when the mention is ambiguous, underspecified, or could reasonably take a different type. 
7. Temporal normalization — the "normalized_value" field:
   - For every entity that denotes a date or time — whether absolute ("24 June", "2026"), relative, or deictic ("Wednesday", "yesterday", "the last week of April", "this weekend", "next year") — resolve it to an ISO-8601 calendar value and put it in "normalized_value". Resolve relative/deictic expressions against REFERENCE_DATE, inferring direction (past vs. future) from the narrative context.
   - Use the granularity the text justifies: "YYYY-MM-DD" for a specific day, or a reduced form ("YYYY-MM", "YYYY") when only a coarser unit is warranted.
   - "normalized_value" holds a single point in time, or the start of a span. When the expression denotes a span (a weekend, a week, a month, a season, an explicit range, etc.), also set "normalized_end" to the end of the span (inclusive) at the justified granularity. For a single point in time, set "normalized_end" to null.
   - NEVER put the normalized value in "text": the original surface form always stays verbatim in "text".
   - For every non-temporal entity, and for any temporal expression you cannot confidently resolve, set both "normalized_value" and "normalized_end" to null.
8. After extracting all entities, collect every unique type label you used. For each, provide: (a) "definition": a concise one-sentence definition describing the category in general terms (what this kind of thing is); and (b) "synonyms": a short list of 2-5 alternative names or near-synonyms for the type (aka labels; e.g. for "Musician": "singer", "recording artist", "performer"). Both must be general and independent of the specific entities in the text. Do not include example instances, and do not include contrastive "distinct from" notes.
9. The set of distinct type labels used in "entities" must be exactly the set of labels defined in "types": every label used on an entity has exactly one definition, and every defined label is used by at least one entity. A label may be used by multiple entities, but is defined only once. The same (text, type) pair must not appear more than once.
10. Return ONLY valid JSON matching the schema below. No explanations, no extra text.
OUTPUT SCHEMA:
{
  "entities": [
    { "text": "entity mention", "type": "type_label", "normalized_value": "YYYY-MM-DD", "normalized_end": "YYYY-MM-DD or null", "confidence": <0-1> }
  ],
  "types": [
    { "type": "type_label", "definition": "A concise general definition of this type.", "synonyms": ["alternative name", "aka", "near-synonym"] }
  ]
}
Note: "normalized_value" and "normalized_end" are null for every non-temporal entity and for unresolvable time expressions; "normalized_end" is set only for spans (null for a single point in time)."""


import re

def _reference_date(text):
    """Parse the 'Date: DD/MM/YYYY' dateline from the article header -> ISO-8601 (YYYY-MM-DD), or None."""
    m = re.search(r"^Date:\s*(\d{2})/(\d{2})/(\d{4})", text, re.MULTILINE)
    if not m:
        return None
    d, mth, y = m.groups()
    return f"{y}-{mth}-{d}"


def extract_entities(article_id):
    """Run Gemini free-form entity+type OIE on one article -> results/<id>/<OUTPUT_NAME>."""
    text = (DATASET_DIR / f"{article_id}.txt").read_text()
    ref_date = _reference_date(text)
    print(f"[{article_id}] ~{len(text)//4} tokens; REFERENCE_DATE={ref_date}; calling {GEMINI_MODEL} ...")
    user_message = (f"REFERENCE_DATE: {ref_date}\n\nTEXT:\n{text}" if ref_date else f"TEXT:\n{text}")
    oie = gemini_json(OIE_ENTITY_SYSTEM_PROMPT, user_message)
    out_dir = RESULTS_DIR / article_id
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / OUTPUT_NAME).write_text(json.dumps(oie, indent=2, ensure_ascii=False))
    n_norm = sum(1 for e in oie.get("entities", []) if e.get("normalized_value"))
    print(f"  latency {last_latency_s:.2f}s | {len(oie.get('entities', []))} entities "
          f"({n_norm} time-normalized), {len(oie.get('types', []))} distinct types -> {out_dir / OUTPUT_NAME}")
    return oie

# oie = extract_entities(ARTICLE_ID)

## 3. Batch — process the selected articles (`ARTICLE_IDS`)

In [5]:
# Runs entity extraction for each article in ARTICLE_IDS (choose them via SELECTED_ARTICLES in the
# config cell). Re-runs overwrite existing outputs. (Relation/triple extraction comes in a later stage.)
import time
nb_parallel.reset_usage()          # the report below covers only the calls THIS run makes
_t0 = time.perf_counter()

for aid in ARTICLE_IDS:
    print("=" * 60, aid)
    with nb_parallel.usage_scope(aid):      # tag this article's calls so the report breaks down per article
        extract_entities(aid)
print(f"\ndone — extracted entities for {len(ARTICLE_IDS)} article(s)")

# --- what this run cost: calls, prompt / output / THINKING tokens, and elapsed time ------------
nb_parallel.usage_report(f"nb2 · OIE entity extraction · {EXPERIMENT}", wall_seconds=time.perf_counter() - _t0,
                         write=RESULTS_DIR / f"_global/token_usage_nb2_oie_entities{_osfx}.json")

============================================================ art1
[art1] ~979 tokens; REFERENCE_DATE=2026-07-03; calling llama33-70b ...
  latency 132.24s | 29 entities (6 time-normalized), 17 distinct types -> /Users/marwah/Documents/Thesis/TriplesExtractionFromWebLLMs/Phase2_LLM_Extraction/results/llama33-70b/art1/two_staged/1_entities/oie_entities_main_val_r1.json
============================================================ art2
[art2] ~1681 tokens; REFERENCE_DATE=2026-07-03; calling llama33-70b ...
  latency 124.92s | 36 entities (8 time-normalized), 13 distinct types -> /Users/marwah/Documents/Thesis/TriplesExtractionFromWebLLMs/Phase2_LLM_Extraction/results/llama33-70b/art2/two_staged/1_entities/oie_entities_main_val_r1.json
============================================================ art3
[art3] ~2813 tokens; REFERENCE_DATE=2026-05-28; calling llama33-70b ...
  latency 167.57s | 37 entities (6 time-normalized), 19 distinct types -> /Users/marwah/Documents/Thesis/TriplesExtractio

{'label': 'nb2 · OIE entity extraction · main',
 'totals': {'calls': 20,
  'prompt_tokens': 45687,
  'output_tokens': 30402,
  'thoughts_tokens': 0,
  'total_tokens': 76089,
  'seconds': 2295.0},
 'wall_seconds': 2295.1772040840006,
 'per_tag': {'art1': {'calls': 1,
   'seconds': 132.2,
   'prompt_tokens': 2153,
   'output_tokens': 1735,
   'thoughts_tokens': 0,
   'total_tokens': 3888},
  'art2': {'calls': 1,
   'seconds': 124.9,
   'prompt_tokens': 2742,
   'output_tokens': 1653,
   'thoughts_tokens': 0,
   'total_tokens': 4395},
  'art3': {'calls': 1,
   'seconds': 167.6,
   'prompt_tokens': 3568,
   'output_tokens': 2209,
   'thoughts_tokens': 0,
   'total_tokens': 5777},
  'art4': {'calls': 1,
   'seconds': 103.1,
   'prompt_tokens': 1691,
   'output_tokens': 1376,
   'thoughts_tokens': 0,
   'total_tokens': 3067},
  'art5': {'calls': 1,
   'seconds': 122.5,
   'prompt_tokens': 2135,
   'output_tokens': 1630,
   'thoughts_tokens': 0,
   'total_tokens': 3765},
  'art8': {'calls': 1

## Run metrics — the numbers this stage produced

`run_metrics.oie` re-reads what nb2 and nb6 wrote: entity and type counts, temporal-normalization
coverage, relation and phrase counts, relations per entity, and the token cost. Saved to
`_global/metrics_oie<_osfx>.json` so the numbers survive the kernel. (nb2 and nb6 share one metrics
file, because the second stage is meaningless without the first — the relation counts are per
*this* entity set.)

In [6]:
# Collect + persist the staged track's open-extraction numbers. Pure reading — no LLM call. Reads
# nb6's relations too when they exist, so re-running this cell after nb6 fills the second half in.
import importlib, run_metrics
importlib.reload(run_metrics)

_m = run_metrics.oie(RESULTS_DIR, ARTICLE_IDS, osfx=_osfx, experiment=EXPERIMENT)
run_metrics.show(_m)

# save() also CLOSES this stage in the run manifest — the record of WHICH run these numbers
# came from: the resolved config, the git commit, the article order, every output file and
# the token spend.  results/<model>/_runs/<run_id>.json  ·  python -m run_manifest list
print("\nsaved ->", run_metrics.save(_m, RESULTS_DIR, _osfx, "binary",
                                     section="2", articles=ARTICLE_IDS))

OIE  ·  binary  ·  experiment 'main'
article         entities  distinct_type  entities_with      relations  distinct_phra  relations_per  mean_confiden
art1                  29             17              6              0              0            0.0           None
art2                  36             13              8              0              0            0.0           None
art3                  37             20              6              0              0            0.0           None
art4                  27              7              2              0              0            0.0           None
art5                  29             15              3              0              0            0.0           None
art8                  37             11              6              0              0            0.0           None
art12                 23             11              8              0              0            0.0           None
art15                 30             17    